# 第24章 世界に公開する：ブラウザで動くあなたのLLM

HTMLとCSSとJavaScriptで、漱石GPTを誰でも使えるWebページにする

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/24-deploy.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。


## 手順1：モデルをJSONに書き出す（Colab）


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
ckpt = "/content/drive/MyDrive/tsukuru-llm/soseki_gpt.pt"
print("チェックポイントはある？", os.path.exists(ckpt))

In [ ]:
import base64, json
import torch

def quantize_rows(w):
    w = w.float()
    if w.dim() == 1:                                   # バイアスやLayerNormはそのまま（float32）
        return {"shape": list(w.shape), "f32": base64.b64encode(w.numpy().tobytes()).decode()}
    scale = w.abs().amax(dim=1).clamp(min=1e-8) / 127  # 行ごとの縮尺
    q = torch.round(w / scale[:, None]).clamp(-127, 127).to(torch.int8)
    return {"shape": list(w.shape),
            "i8": base64.b64encode(q.numpy().tobytes()).decode(),
            "scale": base64.b64encode(scale.numpy().astype("float32").tobytes()).decode()}

ck = torch.load(ckpt, map_location="cpu")
weights = {name: quantize_rows(t) for name, t in ck["model"].items() if not name.endswith("tril")}
with open("soseki-gpt.json", "w", encoding="utf-8") as f:
    json.dump({"config": ck["config"], "chars": ck["chars"], "weights": weights}, f, ensure_ascii=False)
print("書き出したサイズ：", round(os.path.getsize("soseki-gpt.json") / 1e6, 1), "MB")

In [ ]:
from google.colab import files
files.download("soseki-gpt.json")

### Colabで一時的に公開する：Gradio


In [ ]:
!pip install -q gradio
import gradio as gr
from transformers import pipeline

bot = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")

def text_of(content):
    if isinstance(content, list):          # 新しいGradioでは、文字以外の部品のリストのこともある
        return "".join(c.get("text", "") for c in content if isinstance(c, dict))
    return content

def reply(message, history):
    messages = [{"role": h["role"], "content": text_of(h["content"])} for h in history]
    messages.append({"role": "user", "content": text_of(message)})
    out = bot(messages, max_new_tokens=200)
    return out[0]["generated_text"][-1]["content"]

demo = gr.ChatInterface(reply, title="ボンゴ先生に質問しよう")
print(reply("こんにちは！", []))

In [ ]:
demo.launch(share=True)